<div class='heading'>
    <div style='float:left;'><h1>CPSC 8810: Machine Learning for Graphs</h1></div>
     <img style="float: right; padding-right: 10px" width="100" src="https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/images/clemson_paw.png"> </div>
     </div>

**Clemson University**<br>
**Instructor(s):** Aaron Masino <br>

## Lab 7: GNN Recommender Systems
This notebook illustrates implementation of a graph based recommender system using GATConv layers. It illustrates the use of PyTorch Geometric heterogeneous data, [HeteroData](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.HeteroData.html#torch_geometric.data.HeteroData) and heterogenous models. The notebook includes steps to generate a simulated recommendation dataset of users and restaurants each with a distinct feature vector. It also shows how to split the edges in the graph in transductive manner. Model implementation, training, and evaluation are also presented. See [Heterogeneous Graph Learning](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.HeteroData.html#torch_geometric.data.HeteroData) for more more information.

### Learning Objectives
1. Understand the PyG HeteroData data structure
2. Apply GNN layers and the PyG to_hetero method to construct heterogeneous graph models
3. Create recommender systems for bipartite data (e.g., users, items)
4. Evaluate recommender system performance

In [ ]:
!pip3 install torch_geometric

In [ ]:
import torch
import torch.nn as tnn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, to_hetero, SAGEConv
from torch_geometric.seed import seed_everything
from torch_geometric.data import HeteroData

import random
import numpy as np

---
# Part 1: Data Generation
We will generate a synthetic dataset representing a biparitite graph with two node types, **user** and **restaurant**. Relations in the graph will include *(user, recommends, restaurant)* and (restaurant, recommended_by, user)*. The second relation type is simply the inverse of the first and will allow messages to flow from restaurant to user in the message passing framework.

First, let's create the method `generate_user` to simulate users. Each user will have three (3) categorical features:
* age : 8-29, 30-54, 55-74, >75
* location : New York, Philadelphia, Boston, Atlanta
* income: low, medium, high

In [ ]:
def generate_user():
    """Generate a random user feature vector with one-hot encoded attributes."""
    # Age groups: 18-29, 30-54, 55-74, >75
    age = torch.zeros(4)
    age[random.randint(0, 3)] = 1

    # Locations: New York, Philadelphia, Boston, Atlanta
    location = torch.zeros(4)
    location[random.randint(0, 3)] = 1

    # Income levels: low, medium, high
    income = torch.zeros(3)
    income[random.randint(0, 2)] = 1

    return torch.cat([age, location, income])

For convenience, let's add a `decode_user` method that returns the feature semantic labels of a user with the given integer feature vector.

In [ ]:
def decode_user(user_features):
    """Convert user feature tensor to readable text attributes."""
    age_groups = ['18-29', '30-54', '55-74', '>75']
    locations = ['New York', 'Philadelphia', 'Boston', 'Atlanta']
    incomes = ['low', 'medium', 'high']

    age = age_groups[user_features[:4].argmax().item()]
    location = locations[user_features[4:8].argmax().item()]
    income = incomes[user_features[8:].argmax().item()]

    return {'age': age, 'location': location, 'income': income}

Next, let's create the `generate_restaurant` method to simulate restaurants. Each restaurant will have four (4) categorical features:
* location : New York, Philadelphia, Boston, Atlanta
* cost: low, medium, high, very high
* cuisine: american, italian, asian, mexican, indian, bbq
* rating: the average star rating from critics (1-4)

In [ ]:
def generate_restaurant():
    """Generate a random restaurant feature vector with one-hot encoded attributes."""
    # Locations: New York, Philadelphia, Boston, Atlanta
    location = torch.zeros(4)
    location[random.randint(0, 3)] = 1

    # Cost: low, medium, high, very high
    cost = torch.zeros(4)
    cost[random.randint(0, 3)] = 1

    # Cuisine: american, italian, asian, mexican, indian, bbq
    cuisine = torch.zeros(6)
    cuisine[random.randint(0, 5)] = 1

    # Average rating: 1-4
    rating = torch.tensor([random.randint(1, 4)], dtype=torch.float)

    return torch.cat([location, cost, cuisine, rating])

For convenience, let's add a `decode_restaurant` method that returns the feature semantic labels of a restaurant with the given integer feature vector.

In [ ]:
def decode_restaurant(restaurant_features):
    """Convert restaurant feature tensor to readable text attributes."""
    locations = ['New York', 'Philadelphia', 'Boston', 'Atlanta']
    costs = ['Low', 'Medium', 'High', 'Very High']
    cuisines = ['American', 'Italian', 'Asian', 'Mexican', 'Indian', 'BBQ']

    location = locations[restaurant_features[:4].argmax().item()]
    cost = costs[restaurant_features[4:8].argmax().item()]
    cuisine = cuisines[restaurant_features[8:14].argmax().item()]
    rating = int(restaurant_features[14].item())

    return {'location': location, 'cost': cost, 'cuisine': cuisine, 'rating': rating}

Now, let's create method to construct a graph representing our user-restaurant recommendations. The method creates *user* and *restaurant* nodes with randomly selected features. It then assigns edges with the following biased edge probability assignments:
For each user:
1. Draw a random number from a normal distribution using the input mean and standard deviation of the number of recommendations, `mean_recs`. Use the max(1, floor(random number)) as the number of recommended resteraunts for this user.
2. For each recommendation for this user, select a restaurant with the following policy:
    * Select a restaurant in the same location as the user with 70% probability and equal probability for the other locations
    * If the user income is low select a restaurant that is low cost with 50% probability, medium with 25%, and 12.5 for high and very high. Else, if the user income is medium selct a restaruant that is low cost with 25%, medium with 30%, high with 30%, and very high with 15%,. Else, if the user income is high, select the a restaruant with very high cost at 30%, high cost at 30%, medium at 25% and low cost at 15%
    * If the user is under the age of 55 select a restraunt that is one of mexican, asian, indian with 70% probability. If they are 55 or over select one of american or bbq with 60% probability.

The biased policy should result in the user features and restaurant features providing signal about the likelihood that a user recommends a particular restaurant that will be encoded in the node embeddings via graph message passing during training.

Note that the graph includes two edge indices:
`data['user', 'recommends', 'restaurant'].edge_index`
`data['restaurant', 'rev_recommends', 'user'].edge_index`
These are simply the inverse of each other. Although the acutall network is bipartite, in our model we want messages to flow in both directions. We need to use a heterogeneous graph with different edge types because the feature vectors are different for users and restaurants. Hence, we need update them with different projection matrices which will be realized via the GNN layers in our heterogeneous graph model.

Also notice that the method constructs several edge masks. Specifically, it builds a `train_message_mask`, `traing_supervision_mask`, `val_supervision_mask`, and `test_supervision_mask` for each relation type. These form a transductive split of the edges that we'll use below for training and evaluation. In all, there are eight (8) masks for this dataset.

In [ ]:
def generate_user_restaurant_graph(num_restaurants, num_users, mean_recs, std_recs,
                                   mask_fractions=[0.75, 0.15, 0.05, 0.05]):
    """Generate a bipartite user-restaurant recommendation graph with edge masks."""
    # Generate all users and restaurants
    users = torch.stack([generate_user() for _ in range(num_users)])
    restaurants = torch.stack([generate_restaurant() for _ in range(num_restaurants)])

    # Build edges with biased selection
    edge_list = []

    for user_idx in range(num_users):
        user_features = users[user_idx]

        # Determine number of recommendations for this user
        num_recs = max(1, int(np.floor(np.random.normal(mean_recs, std_recs))))

        # Get user attributes
        user_age_idx = user_features[:4].argmax().item()
        user_loc_idx = user_features[4:8].argmax().item()
        user_income_idx = user_features[8:].argmax().item()

        for _ in range(num_recs):
            # Sample restaurant with biased policy

            # Location bias: 70% same location, 10% each for others
            loc_probs = [0.1, 0.1, 0.1, 0.1]
            loc_probs[user_loc_idx] = 0.7
            target_loc = np.random.choice(4, p=loc_probs)

            # Cost bias based on income
            if user_income_idx == 0:  # low income
                cost_probs = [0.5, 0.25, 0.125, 0.125]
            elif user_income_idx == 1:  # medium income
                cost_probs = [0.25, 0.3, 0.3, 0.15]
            else:  # high income
                cost_probs = [0.15, 0.25, 0.3, 0.3]
            target_cost = np.random.choice(4, p=cost_probs)

            # Cuisine bias based on age
            if user_age_idx < 2:  # under 55
                cuisine_probs = [0.1, 0.1, 0.233, 0.233, 0.234, 0.1]
            else:  # 55 or over
                cuisine_probs = [0.3, 0.1, 0.1, 0.1, 0.1, 0.3]
            target_cuisine = np.random.choice(6, p=cuisine_probs)

            # Find matching restaurants
            candidates = []
            for rest_idx in range(num_restaurants):
                rest_features = restaurants[rest_idx]
                rest_loc = rest_features[:4].argmax().item()
                rest_cost = rest_features[4:8].argmax().item()
                rest_cuisine = rest_features[8:14].argmax().item()

                if rest_loc == target_loc and rest_cost == target_cost and rest_cuisine == target_cuisine:
                    candidates.append(rest_idx)

            # Select restaurant (random from candidates, or fallback to any)
            if candidates:
                rest_idx = random.choice(candidates)
            else:
                rest_idx = random.randint(0, num_restaurants - 1)

            edge_list.append([user_idx, rest_idx])

    # Create edge tensors
    edge_index = torch.tensor(edge_list, dtype=torch.long).t()
    num_edges = edge_index.size(1)

    # Create symmetric edge masks
    indices = torch.randperm(num_edges)

    # Calculate split points
    train_msg_end = int(mask_fractions[0] * num_edges)
    train_sup_end = train_msg_end + int(mask_fractions[1] * num_edges)
    val_sup_end = train_sup_end + int(mask_fractions[2] * num_edges)

    # Initialize masks
    train_message_mask = torch.zeros(num_edges, dtype=torch.bool)
    train_supervision_mask = torch.zeros(num_edges, dtype=torch.bool)
    val_supervision_mask = torch.zeros(num_edges, dtype=torch.bool)
    test_supervision_mask = torch.zeros(num_edges, dtype=torch.bool)

    # Assign edges to masks
    train_message_mask[indices[:train_msg_end]] = True
    train_supervision_mask[indices[train_msg_end:train_sup_end]] = True
    val_supervision_mask[indices[train_sup_end:val_sup_end]] = True
    test_supervision_mask[indices[val_sup_end:]] = True

    # Create PyG HeteroData object with bidirectional edges and masks
    data = HeteroData()
    data['user'].x = users
    data['restaurant'].x = restaurants
    data['user', 'recommends', 'restaurant'].edge_index = edge_index
    data['restaurant', 'rev_recommends', 'user'].edge_index = edge_index.flip(0)

    # Add symmetric masks to both relations
    # NOTE: in this case the masks are identical for both directions because if user u recommends restaurant r,
    # then restaurant r is recommended by user u and the edge_index tensors are symmetric. In an arbitrary graph,
    # the masks would likely differ.
    data['user', 'recommends', 'restaurant'].train_message_mask = train_message_mask
    data['user', 'recommends', 'restaurant'].train_supervision_mask = train_supervision_mask
    data['user', 'recommends', 'restaurant'].val_supervision_mask = val_supervision_mask
    data['user', 'recommends', 'restaurant'].test_supervision_mask = test_supervision_mask

    data['restaurant', 'rev_recommends', 'user'].train_message_mask = train_message_mask
    data['restaurant', 'rev_recommends', 'user'].train_supervision_mask = train_supervision_mask
    data['restaurant', 'rev_recommends', 'user'].val_supervision_mask = val_supervision_mask
    data['restaurant', 'rev_recommends', 'user'].test_supervision_mask = test_supervision_mask

    return data

Recall, that in the transductive edge split approach, message passing is done with only the `train_message` edges during training. During validation, message passing happens on the `train_message` + `train_supervision` edges. Finally, during test, message passing happens on the `train_message` + `train_supervision` + `val_supervision` edges. It will be convenient to have the method below, which produces an edge_index_dict representing the message passing edges for a given stage of the training and evaluation pipeline. The method takes as input a PyG `HeteroData` object, a variable `split` that is one of {'train', 'val', 'test'} and returns an `edge_dict` representing all of the edges to be used for message passing.

In [ ]:
def get_message_passing_edge_dict(data, split):
    """Get edge_index_dict for message passing based on split."""
    edge_dict = {}

    if split == 'train':
        # Only train_message_mask edges
        for edge_type in data.edge_types:
            mask = data[edge_type].train_message_mask
            edge_dict[edge_type] = data[edge_type].edge_index[:, mask]

    elif split == 'val':
        # train_message_mask + train_supervision_mask edges
        for edge_type in data.edge_types:
            mask = data[edge_type].train_message_mask | data[edge_type].train_supervision_mask
            edge_dict[edge_type] = data[edge_type].edge_index[:, mask]

    elif split == 'test':
        # All edges except test_supervision_mask
        for edge_type in data.edge_types:
            mask = ~data[edge_type].test_supervision_mask
            edge_dict[edge_type] = data[edge_type].edge_index[:, mask]

    else:
        raise ValueError(f"Invalid split: {split}. Must be 'train', 'val', or 'test'.")

    return edge_dict

Let's generate a dataset.

In [ ]:
data = generate_user_restaurant_graph(num_restaurants=50, num_users=5000, mean_recs=5, std_recs=2)

Let's examine the dataset structure.

In [ ]:
print(data)
print("\nLet's look at some edges and node features:")
print(data['user','recommends','restaurant'].edge_index[:, :10])  # Show first 10 edges
print(data['restaurant','rev_recommends','user'].edge_index[:, :10])  # Show first 10 edges
print("\nLet's decode a user and a restaurant feature vector:")
print(decode_user(data['user'].x[0]))
print(decode_restaurant(data['restaurant'].x[0]))

Let's take a closer look at the feature and edge dictionaries create for the `HeteroData` object.

In [ ]:
# Let's look at the node and edge feature dictionaries
print(data.x_dict)
data.edge_index_dict

In [ ]:
# Let's pull the training message passing edge_index_dict
get_message_passing_edge_dict(data, 'train')

---
# Part 2: Neural Graph Collaborative Filtering Model Development

## 2.1 Model Architecture
We're going to build a collaborative filtering model that is an extension of the neural graph collaborative filtering (NGCF) method we saw in class. Recall, that approach used GCN layers, but did not include node features. Here, we use the same concept, but we will use the node features by constructing a heterogeneous graph.

First, let's construct the model arhictecture. We can express the model as a homogeneous graph and then use the PyG `to_hetero` method to convert it to a heterogeneous model. Note that only some GNN layers are supported by the `to_hetero` method, see the [cheatsheet](https://pytorch-geometric.readthedocs.io/en/latest/notes/cheatsheet.html). We will use GATConv layers. We will develop a simple model with only two layers for illustration. Note that in the first GATConv layer, we pass a tuple `(-1,-1)` as the input_channel size. This tells the GATConv __init__ to use lazy instantiation and to set the weight matrix sizes based on the input feature sizes of the two node types in the bipartite graph. Not that for each realtion type, two matrices are required because the input feature dimensions are of different sizes and have different semantics. Consider the user-recommends-restaurant relation. For a given restaurant node, the embedding will be updated by summing a tranformation W0*h of the restaurant feature vector (or embedding at next layer) and an aggregation of the transformed neighbor nodes (users) W1*h. As restaurants and users have different feature vectors, W0 and W1 necessarily have different shapes. Assuming the embedding dimension is d, W0 is d X 3 (each user has 3 features), and W1 is d X 4 (each restaurant has 4 features). The resulting restaurant embedding is d X 1.

In [ ]:
class BipartiteEncoder(tnn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = GATConv((-1, -1), hidden_channels, add_self_loops=False)
        self.conv2 = GATConv((hidden_channels, hidden_channels), hidden_channels, add_self_loops=False)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.conv2(x, edge_index)

        return x

We can now convert this model to the corresponding heterogeneous model by passing the metadata to the `to_hetero` method.

In [ ]:
model = BipartiteEncoder(32)
node_types = ['user', 'restaurant']
edge_types = [('user','recommends','restaurant'),
              ('restaurant','rev_recommends','user')]
metadata = (node_types, edge_types)
model = to_hetero(model, metadata)

In [ ]:
out = model(data.x_dict, data.edge_index_dict)
print(out['user'].shape)
out['restaurant'].shape

## 2.2 Model Training

To train the model, we will need positive and negative edge samples. The positive samples were generated when we created the data. We will sample negative samples using the method below. It takes as input the `data` which is the *HeteroData* object representing our network, the `edge_index_dict` which contains our set of positive samples for each edge type - these will be our train, val, or test supervision edges. The `exclude_edge_index_dict` contains edges that should not be used as negative samples. Ideally, this should be all edges in the graph, but that may be computationally infeasible. To form the negative samples, we consider each postive sample of the form *(head, relation, tail)* and corrupt the tail. That is, we form examples *(head, relation, tailX)* where *tailX != tail* and *(head, relation, tailX)* is not in the `exclude_edge_index_dict[relation]` edges.

In [ ]:
def sample_negative_edges(data, edge_index_dict, exclude_edge_index_dict):
    """Sample negative edges by corrupting tail nodes, avoiding edges in exclude set."""
    neg_edge_dict = {}

    for edge_type in edge_index_dict.keys():
        edge_index = edge_index_dict[edge_type]
        num_edges = edge_index.size(1)

        # Get head and tail node types
        head_type, _, tail_type = edge_type

        # Get number of tail nodes
        num_tail_nodes = data[tail_type].x.size(0)

        # Create set of existing edges to exclude
        exclude_edges = exclude_edge_index_dict[edge_type]
        exclude_set = set()
        for i in range(exclude_edges.size(1)):
            exclude_set.add((exclude_edges[0, i].item(), exclude_edges[1, i].item()))

        # Sample negative tail nodes for each edge
        neg_tails = []
        for i in range(num_edges):
            head = edge_index[0, i].item()

            # Sample until we find a tail that doesn't exist in exclude_set
            while True:
                neg_tail = torch.randint(0, num_tail_nodes, (1,)).item()
                if (head, neg_tail) not in exclude_set:
                    neg_tails.append(neg_tail)
                    break

        # Create negative edges (keep head, replace tail)
        neg_tail_tensor = torch.tensor(neg_tails, dtype=torch.long)
        neg_edge_index = torch.stack([edge_index[0], neg_tail_tensor], dim=0)

        neg_edge_dict[edge_type] = neg_edge_index

    return neg_edge_dict

Above, we created the `BipartiteEncoder` which uses GNN layers to update the node embeddings. Given two node embeddings, `e1` and `e2`, we need a method that provides a scalar score that we can use to decide if the edge (e1, relation, e2) should exist. For example, given the embedding `eu` for user *u* and `er` for restaurant `r`, we need a score to decide if (u, recommends, r) should exist. We will use the dot product of the embeddings, `dot(eu, er)`. Given a set of restaurants and a user, we can then simply rank the restaurants by the dot product scores.

In [ ]:
def decode_edges(node_embeddings, edge_index_dict):
    """Compute scores for edges as dot product of head and tail embeddings."""
    scores_dict = {}

    for edge_type in edge_index_dict.keys():
        edge_index = edge_index_dict[edge_type]

        # Get head and tail node types
        head_type, _, tail_type = edge_type

        # Get embeddings for head and tail nodes
        head_emb = node_embeddings[head_type][edge_index[0]]  # [num_edges, hidden_dim]
        tail_emb = node_embeddings[tail_type][edge_index[1]]  # [num_edges, hidden_dim]

        # Compute scores as dot product
        scores = (head_emb * tail_emb).sum(dim=1)  # [num_edges]

        scores_dict[edge_type] = scores

    return scores_dict

We will also need a loss function. Below, we implement the Bayesian Personalized Rank loss as presented in class.

In [ ]:
def bpr_loss(positive_scores_dict, negative_scores_dict_list):
    """Compute Bayesian Personalized Ranking loss for link prediction."""
    total_loss = 0.0

    for edge_type in positive_scores_dict.keys():
        # Get positive scores
        pos_scores = positive_scores_dict[edge_type]  # [num_edges]
        num_pos_edges = pos_scores.size(0)
        k = len(negative_scores_dict_list)

        # Compute BPR loss for each negative sample
        for neg_scores_dict in negative_scores_dict_list:
            neg_scores = neg_scores_dict[edge_type]  # [num_edges]

            # BPR loss: -log(sigmoid(pos_score - neg_score))
            loss = -F.logsigmoid(pos_scores - neg_scores).sum()
            total_loss += loss

        # Normalize by k * num_pos_edges
        total_loss = total_loss / (k * num_pos_edges)

    return total_loss

Finally we need training and evaluation methods. The `train_epoch` method below trains the model on one pass of the data. Notice that in the forward call to the model, `node_embeddings = model(data.x_dict, train_msg_edges)`, that the a dictionary is passed in for the node features and the edge index information. The `x_dict` is a dictionary where keys are the node types and the values are the node feature tensor. The edge index dictionary has keys that are triples indicating the relation type, such as `('user', 'recommends', 'restaurant')` and the values are the *edge_index* for that key. Also notice, that we do not pass in all edges, but rather only the training message passing edges, `train_msg_edges`. Finally, not that to compute the loss, we first compute the scores, `pos_scores` on the postive samples and on `num_neg_samples` per postive sample.

In [ ]:
def train_epoch(model, data, optimizer, num_neg_samples=5):
    """Train the model for one epoch."""
    model.train()
    optimizer.zero_grad()

    # Get train message passing edges
    train_msg_edges = get_message_passing_edge_dict(data, 'train')

    # Forward pass with train message edges
    node_embeddings = model(data.x_dict, train_msg_edges)

    # Get train supervision edges for loss computation
    train_sup_edges = {}
    for edge_type in data.edge_types:
        mask = data[edge_type].train_supervision_mask
        train_sup_edges[edge_type] = data[edge_type].edge_index[:, mask]

    # Get excluded edges (train_message + train_supervision)
    exclude_edges = {}
    for edge_type in data.edge_types:
        mask = data[edge_type].train_message_mask | data[edge_type].train_supervision_mask
        exclude_edges[edge_type] = data[edge_type].edge_index[:, mask]

    # Sample negative edges
    neg_edge_dicts = []
    for _ in range(num_neg_samples):
        neg_edges = sample_negative_edges(data, train_sup_edges, exclude_edges)
        neg_edge_dicts.append(neg_edges)

    # Compute scores
    pos_scores = decode_edges(node_embeddings, train_sup_edges)
    neg_scores_list = [decode_edges(node_embeddings, neg_edges) for neg_edges in neg_edge_dicts]

    # Compute loss
    loss = bpr_loss(pos_scores, neg_scores_list)

    # Backward pass
    loss.backward()
    optimizer.step()

    return loss.item()

def train(model, data, optimizer, num_epochs, num_neg_samples=5):
    """Train the model for multiple epochs with validation."""
    for epoch in range(1, num_epochs + 1):
        train_loss = train_epoch(model, data, optimizer, num_neg_samples)
        val_loss = validate(model, data, num_neg_samples)
        print(f'Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    return model

In [ ]:
@torch.no_grad()
def validate(model, data, num_neg_samples=5):
    """Validate the model."""
    model.eval()

    # Get validation message passing edges (train_message + train_supervision)
    val_msg_edges = get_message_passing_edge_dict(data, 'val')

    # Forward pass with validation message edges
    node_embeddings = model(data.x_dict, val_msg_edges)

    # Get validation supervision edges for loss computation
    val_sup_edges = {}
    for edge_type in data.edge_types:
        mask = data[edge_type].val_supervision_mask
        val_sup_edges[edge_type] = data[edge_type].edge_index[:, mask]

    # Get excluded edges (train_message + train_supervision + val_supervision)
    exclude_edges = {}
    for edge_type in data.edge_types:
        mask = (data[edge_type].train_message_mask |
                data[edge_type].train_supervision_mask |
                data[edge_type].val_supervision_mask)
        exclude_edges[edge_type] = data[edge_type].edge_index[:, mask]

    # Sample negative edges
    neg_edge_dicts = []
    for _ in range(num_neg_samples):
        neg_edges = sample_negative_edges(data, val_sup_edges, exclude_edges)
        neg_edge_dicts.append(neg_edges)

    # Compute scores
    pos_scores = decode_edges(node_embeddings, val_sup_edges)
    neg_scores_list = [decode_edges(node_embeddings, neg_edges) for neg_edges in neg_edge_dicts]

    # Compute loss
    loss = bpr_loss(pos_scores, neg_scores_list)

    return loss.item()

Now let's create a method to evaluate test performance on the test edges. We will assess performance using the *Hits@K* method. You should try working through the code to understand how this metric is computes. Here are some details:
1. There are P positive samples for each edge type
2. We sample num_neg_samples (100) sets of negative edges. Each set has P negative edges (one negative per positive). So:
    * neg_edge_dicts[0] = first set of P negative edges (one per positive)
    * neg_edge_dicts[1] = second set of P negative edges (one per positive)
    * ...
    * neg_edge_dicts[99] = 100th set of P negative edges (one per positive)
3. For each edge type, neg_score_tensor has shape [P, 100] where:
    * neg_score_tensor[i, j] = score for the j-th negative sample corresponding to positive sample i
4. Noting that pos_score.unsqueeze(1) is a tensor with shape [num_positive samples, 1], the line `ranks = (neg_score_tensor >= pos_score.unsqueeze(1)).sum(dim=1) + 1` finds the number of negative samples with score >= to the postive sample, adding 1 to make it 1 based instead of zero based
5. The hits_at_k computation uses the ranks tensor to get the booleans for the condition ranks<k and takes mean, which is just the number of positive samples with a rank better than or equal to k.

In [ ]:
@torch.no_grad()
def test(model, data, k_vals=[1, 5, 10], num_neg_samples=100):
    """Test the model using Hits@K metric."""
    model.eval()

    # Get test message passing edges (all except test_supervision)
    test_msg_edges = get_message_passing_edge_dict(data, 'test')

    # Forward pass with test message edges
    node_embeddings = model(data.x_dict, test_msg_edges)

    # Get test supervision edges for evaluation
    test_sup_edges = {}
    for edge_type in data.edge_types:
        mask = data[edge_type].test_supervision_mask
        test_sup_edges[edge_type] = data[edge_type].edge_index[:, mask]

    # Get excluded edges (all edges in the graph)
    exclude_edges = data.edge_index_dict

    # Sample negative edges
    neg_edge_dicts = []
    for _ in range(num_neg_samples):
        neg_edges = sample_negative_edges(data, test_sup_edges, exclude_edges)
        neg_edge_dicts.append(neg_edges)

    # Compute scores
    pos_scores = decode_edges(node_embeddings, test_sup_edges)
    neg_scores_list = [decode_edges(node_embeddings, neg_edges) for neg_edges in neg_edge_dicts]

    # Compute Hits@K for each relation type
    results = {}
    for edge_type in test_sup_edges.keys():
        pos_score = pos_scores[edge_type]  # [num_pos_edges]

        # Stack negative scores [num_pos_edges, num_neg_samples]
        neg_score_list = [neg_scores[edge_type] for neg_scores in neg_scores_list]
        neg_score_tensor = torch.stack(neg_score_list, dim=1)

        # For each positive edge, count how many negatives have lower scores
        # pos_score: [num_pos_edges], neg_score_tensor: [num_pos_edges, num_neg_samples]
        ranks = (neg_score_tensor >= pos_score.unsqueeze(1)).sum(dim=1) + 1  # [num_pos_edges]

        # Compute Hits@K
        hits_at_k = {}
        for k in k_vals:
            hits_at_k[k] = (ranks <= k).float().mean().item()

        results[edge_type] = hits_at_k

    # Print results
    print("Test Results:")
    for edge_type in results.keys():
        print(f"  {edge_type}:")
        for k in k_vals:
            print(f"    Hits@{k}: {results[edge_type][k]:.4f}")

    return results

Finally, let's train and evaluate a model.

In [ ]:
# Initialize model
model = BipartiteEncoder(hidden_channels=64)
node_types = ['user', 'restaurant']
edge_types = [('user', 'recommends', 'restaurant'),
              ('restaurant', 'rev_recommends', 'user')]
metadata = (node_types, edge_types)
model = to_hetero(model, metadata)

# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Train the model
model = train(model, data, optimizer, num_epochs=50, num_neg_samples=5)

In [ ]:
# Test the model
test_results = test(model, data, k_vals=[1, 5, 10, 20], num_neg_samples=100)

The model performs reasonably well considering how simple it is. However, there are a few factors besides the model contributing to the strong performance:

1. Strong feature correlations: The biased sampling policy creates meaningful patterns that he GNN can learns from the node features.
2. Rich connectivity: With 5000 users, ~5 recommendations each, and only 50 restaurants, there's substantial connectivity (~25k edges). This gives the model lots of signal about which user types connect to which restaurant types.

Scaling to thousands of restaurants with sparser, noisier features would likely be much more challenging.